# AlphaGenome Cancer Variant Inspector & Visualization Cookbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DixitPatel/alphagenome_sidecar/blob/main/colabs/cancer_variant_cookbook.ipynb)

> **Meta Global AI Developer Hackathon 2026** — *Build for Your World*
> Companion cookbook for **cBioPortal & Genome Nexus AlphaGenome Integration**

This interactive notebook allows cancer researchers to inspect **1 Mb base-pair resolution predictions** from Google DeepMind’s AlphaGenome model for somatic variants identified in cBioPortal.

### What this Cookbook Does:
1. **Centers a 1 Mb sequence window** around any cancer variant (SNV or indel).
2. **Extracts strand-specific RNA-seq coverage** for the patient’s matched cancer tissue (e.g. Pancreas ).
3. **Dynamically detects gene strand** ( vs ) to prevent false-negative flatlines.
4. **Plots Overlaid REF vs. ALT tracks** to visualize transcriptional repression, promoter shifts, and splicing disruption.


In [ ]:
# @title 1. Install & Import AlphaGenome Dependencies
# @markdown Installs the official AlphaGenome SDK and visualization libraries.
!pip install -q alphagenome matplotlib pandas pyarrow

import os
import pandas as pd
import matplotlib.pyplot as plt

from alphagenome.data import genome, transcript
from alphagenome.data import gene_annotation
from alphagenome.models.v1 import dna_client
from alphagenome.visualization import plot_components
from alphagenome import colab_utils

print("✅ AlphaGenome SDK successfully imported!")


In [ ]:
# @title 2. Authenticate with AlphaGenome API Key
# @markdown Enter your non-commercial AlphaGenome API key from [alphagenome.google/api](https://alphagenome.google/api).
API_KEY = "" # @param {type:"string"}

if not API_KEY.strip():
    # Fallback to interactive Colab prompt if left blank
    API_KEY = colab_utils.get_api_key()

dna_model = dna_client.create(API_KEY.strip())
output_metadata = dna_model.output_metadata(organism=dna_client.Organism.HOMO_SAPIENS)
print("✅ AlphaGenome model client initialized!")


In [ ]:
# @title 3. Load GENCODE Gene Annotations (One-Time Setup)
# @markdown Downloads GENCODE v46 annotations required for exon/intron transcript mapping.
gtf_url = (
    "https://storage.googleapis.com/alphagenome/reference/gencode/"
    "hg38/gencode.v46.annotation.gtf.gz.feather"
)
print("⏳ Loading GENCODE transcript annotations (takes ~30s on first load)...")
gtf = pd.read_feather(gtf_url)
gtf_coding = gene_annotation.filter_protein_coding(gtf)
gtf_supported = gene_annotation.filter_transcript_support_level(gtf_coding, ["1"])
gtf_longest = gene_annotation.filter_to_longest_transcript(gtf_supported)
longest_extractor = transcript.TranscriptExtractor(gtf_longest)
print("✅ GENCODE transcript annotations indexed and ready!")


In [ ]:
# @title 4. Cancer Variant & Target Tissue Parameters
# @markdown Paste the parameters generated by the **Meta Muse Co-Pilot** in cBioPortal:

VARIANT_STRING = "chr18:48604696:GAA>G" # @param {type:"string"}
GENE_SYMBOL = "SMAD4" # @param {type:"string"}
TISSUE_UBERON = "UBERON:0001264" # @param {type:"string"}
PADDING_KB = 15 # @param {type:"integer"}
INCLUDE_SPLICING = True # @param {type:"boolean"}

print(f"Target: {GENE_SYMBOL} ({VARIANT_STRING}) in {TISSUE_UBERON}")


In [ ]:
# @title 5. Execute AlphaGenome Prediction (REF vs. ALT 1 Mb Window)
variant = genome.Variant.from_str(VARIANT_STRING)

# Dynamically center 1 Mb window on variant position
window_start = max(0, variant.position - 524_288)
window_end = window_start + dna_client.SEQUENCE_LENGTH_1MB
interval = genome.Interval(variant.chromosome, window_start, window_end)

requested_outputs = {dna_client.OutputType.RNA_SEQ}
if INCLUDE_SPLICING:
    requested_outputs.add(dna_client.OutputType.SPLICE_SITES)

print(f"Querying AlphaGenome for 1 Mb interval {interval.chromosome}:{interval.start}-{interval.end}...")
outputs = dna_model.predict_variant(
    interval=interval,
    variant=variant,
    requested_outputs=requested_outputs,
    ontology_terms=[TISSUE_UBERON],
)
print("✅ Predictions successfully computed for REF and ALT sequences!")


In [ ]:
# @title 6. Visualize Overlaid REF vs. ALT RNA Expression
# Step 1: Detect gene strand from GENCODE to avoid blank-signal flatlines
longest_transcripts = longest_extractor.extract(interval)
target_tx = [t for t in longest_transcripts if t.info.get("gene_name") == GENE_SYMBOL]

is_pos_strand = True
if target_tx and target_tx[0].strand == "-":
    is_pos_strand = False

strand_str = "+" if is_pos_strand else "-"
print(f"Gene {GENE_SYMBOL} is on the {strand_str} strand.")

# Filter to the active strand
if is_pos_strand:
    ref_rna = outputs.reference.rna_seq.filter_to_nonnegative_strand()
    alt_rna = outputs.alternate.rna_seq.filter_to_nonnegative_strand()
else:
    ref_rna = outputs.reference.rna_seq.filter_to_nonpositive_strand()
    alt_rna = outputs.alternate.rna_seq.filter_to_nonpositive_strand()

# Step 2: Calculate zoom interval (guarantees both variant and gene are visible)
gene_interval = gene_annotation.get_gene_interval(gtf, gene_symbol=GENE_SYMBOL)
zoom_start = min(variant.position - (PADDING_KB * 1000), gene_interval.start - 1000)
zoom_end = max(variant.position + (PADDING_KB * 1000), gene_interval.end + 1000)
zoom_interval = genome.Interval(variant.chromosome, max(0, zoom_start), zoom_end)

# Step 3: Render OverlaidTracks
ref_alt_colors = {"REF": "dimgrey", "ALT": "crimson"}

plot = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(longest_transcripts),
        plot_components.OverlaidTracks(
            tdata={"REF": ref_rna, "ALT": alt_rna},
            colors=ref_alt_colors,
            ylabel_template="RNA_SEQ ({strand})
{biosample_name}",
        ),
    ],
    annotations=[plot_components.VariantAnnotation([variant])],
    interval=zoom_interval,
    title=f"{GENE_SYMBOL} ({VARIANT_STRING}) — Predicted RNA Expression in {TISSUE_UBERON}",
)
plt.show()


In [ ]:
# @title 7. Splicing Disruption Track (REF vs. ALT Splice Sites)
if INCLUDE_SPLICING and hasattr(outputs.reference, "splice_sites"):
    ref_splice = outputs.reference.splice_sites
    alt_splice = outputs.alternate.splice_sites

    plot = plot_components.plot(
        [
            plot_components.TranscriptAnnotation(longest_transcripts),
            plot_components.OverlaidTracks(
                tdata={"REF": ref_splice, "ALT": alt_splice},
                colors={"REF": "steelblue", "ALT": "darkorange"},
                ylabel_template="Splice Sites
{name}",
            ),
        ],
        annotations=[plot_components.VariantAnnotation([variant])],
        interval=zoom_interval,
        title=f"{GENE_SYMBOL} — Predicted Splice Site Disruption (REF vs. ALT)",
    )
    plt.show()
else:
    print("Splicing track omitted or not requested.")
